In [1]:
# @title Setup — Run this first
import os

if not os.path.exists('/content/MS_spectra_encoding'):
    !git clone https://github.com/Alexander-Sol/MS_spectra_encoding.git /content/MS_spectra_encoding

%cd /content/MS_spectra_encoding

# Install packages not pre-installed on Colab
!pip install -q pyteomics==4.6.1 spectrum-utils==0.4.2 rapidhash==0.1.0 mmh3

# TODO: download data file from Google Drive once hosted
# !gdown <FILE_ID> -O Data/04-17-23_CA_Tryp_HCD_10min-calib.mzML

fatal: could not create leading directories of '/content/MS_spectra_encoding': Permission denied
[Errno 2] No such file or directory: '/content/MS_spectra_encoding'
/home/avni/source/repos/MS_spectra_encoding


## Goals:

In this notebook, we'll be exploring the following three core concepts:

1. Why we need to modify our idea of positional encoding to encode DIA data
2. How to build an "augmented spectrum" data structure that captures contextual information from neighboring MS1 and MS2 scans
3. How to encode the resulting 4-tuple peaks using sinusoidal positional encoding on real DIA data

In [2]:
# @title Run this cell to import all necessary packages

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pyteomics import mzml
from util import *

## 4. Cascadia: Positional Embedding of Augmented Spectra

### 4.1 The Problem: What if one spectrum isn't enough?

In the previous notebook, we saw how Casanovo predicts peptide sequences over DDA data, whereas Cascadia predicts peptide sequences over **DIA** data. So what happens when:
- Your peptide's signal is spread across multiple MS2 cycles (like in DIA)?
- or the MS1 isotope pattern holds crucial information about charge state and mass?
- or neighboring spectra contain corroborating evidence that could improve predictions?

Our motivation here is to leverage all contextual information available to make a better prediction. With this in mind, we can better understand the reasoning behind the "Augmented Spectra" concept introduced in Cascadia.

<details>
<summary><strong>Data-Dependent Acquisition (DDA) vs. Data-Independent Acquisition (DIA)</strong></summary>

<img src='https://github.com/Alexander-Sol/MS_spectra_encoding/blob/Colab/Images/DDA%20vs.%20DIA.png?raw=1' width=500>

**DDA**: Each fragmentation spectrum corresponds to a single peptide.

At every MS1 scan, the instrument selects the top k most intense precursor ions. Only these precursors are isolated, fragmented, and recorded as MS2 spectra. This produces clean MS2 spectra, each representing one peptide, but at the cost of completeness — not all precursors are selected.

**DIA**: A single peptide is reconstructed from multiple time-adjacent spectra collected across wide, overlapping m/z windows.

Instead of selecting specific precursors, DIA fragments every ion within a set of predefined m/z windows. These windows often overlap to avoid losing peptides at boundaries. A single peptide typically contributes fragments to many consecutive MS2 spectra.

This produces more complex MS2 spectra, but the data is more complete because _all_ precursors are fragmented. For a visual of what I mean here, see the first few seconds of the video below.

---

</details>

### Constructing an Augmented Spectrum

Cascadia has a three step process for building an augmented spectrum from MS1 and MS2 data:

**(1) Select a central MS2 scan**

**(2) Include neighboring MS2 scans within a chosen width, and**

**(3) Identify the corresponding MS1 scans in order to build the "augmented spectrum"**

### 4.2 Building an Augmented Spectrum Data Structure

Here are the elements of an augmented spectrum:

| Field | Description |
|-------|-------------|
| m/z | The mass-to-charge ratio of the peak |
| Intensity | Peak intensity, normalized to [0, 1] |
| Width | Relative position: `0` = central scan, `+1` = one scan forward, `-1` = one scan backward |
| Level | `2` = MS2 (fragment ions), `1` = MS1 (precursor/isotope peaks) |

A full augmented spectrum is a collection of peaks represented as 4-tuples: **(m/z, intensity, width, level)**.

---

#### Cascadia's Augmented Spectra: An Example

In our video, we used width = 2, meaning we selected 2 MS2 scans on each side of the central scan. The example below from [Cascadia's documentation](https://cascadia.readthedocs.io/en/latest/file_formats.html) uses width = 1 (1 scan on each side), which makes it easier to trace through.


|m/z                    | intensity                |width | level|
|-----------------------|--------------------------|------|------|
|185.07986450195312      | 0.666241404428991       | 0     | 2
|367.8453063964844       | 0.4087404090593467      | 0     | 2
|400.98358154296875      | 0.8838386047632094      | 0     | 2
|401.1849060058594       | 0.4504917813994659      | 0     | 2
|401.2586669921875       | 0.42375381912879023     | 0     | 2
|401.9834289550781       | 0.6479885582525154      | 0     | 2
|418.99261474609375      | 0.6245163448036832      | 0     | 2
|606.7568359378328       | 0.41198908195979644     | 0     | 2
|606.756835937573        | 0.41198908195979644     | 0     | 2
|843.2614135742188       | 0.41591027114618606     | 0     | 2
|946.1708374023438       | 0.430206516710194       | 0     | 2
|185.07986450195312      | 0.666241404428991       | 1     | 2
|367.8453063964844       | 0.4087404090593467      | 1     | 2
|400.98358154296875      | 0.8838386047632094      | 1     | 2
|401.1849060058594       | 0.4504917813994659      | 1     | 2
|401.2586669921875       | 0.42375381912879023     | 1     | 2
|401.9834289550781       | 0.6479885582525154      | 1     | 2
|418.99261474609375      | 0.6245163448036832      | 1     | 2
|185.07986450195312      | 0.666241404428991       | -1    | 2
|367.8453063964844       | 0.4087404090593467      | -1    | 2
|400.98358154296875      | 0.8838386047632094      | -1    | 2
|401.1849060058594       | 0.4504917813994659      | -1    | 2
|401.2586669921875       | 0.42375381912879023     | -1    | 2
|401.9834289550781       | 0.6479885582525154      | -1    | 2
|418.99261474609375      | 0.6245163448036832      | -1    | 2
|606.7568359375737       | 0.41198908195979644     | -1    | 2
|843.2614135742188       | 0.41591027114618606     | -1    | 2
|946.1708374023438       | 0.430206516710194       | -1    | 2
|400.1812744140625       | 0.2918983249625358      | 0     | 1
|400.3501892089844       | 0.3196363715025925      | 0     | 1
|401.2091979980469       | 0.34867104752184375     | 0     | 1
|403.2320556640625       | 0.30672175589489503     | 0     | 1
|402.2713624687576       | 0.2951177103313615      | -1    | 1
|403.17193603515625      | 0.362321942870444       | -1    | 1


<br/>

> Note that all of the data here is exploded into peaks, so each row corresponds to one peak from a given MS2 (and its corresponding MS1) scan.

**The table above is just one augmented spectrum**, not several. It represents a single event. We've chosen a single central MS2 scan (at width = 0) plus its temporal neighbors.

Here's how to read the groups:

| What you see | What it means |
|---|---|
| Rows with `width=0, level=2` | Fragment peaks from the **central** MS2 scan|
| Rows with `width=+1, level=2` | Fragment peaks from the **next** occurrence of the same isolation window|
| Rows with `width=-1, level=2` | Fragment peaks from the **previous** occurrence|
| Rows with `level=1` | Precursor/isotope peaks from the **MS1** scan of that cycle |

Another corollary we can infer is the fact that the same m/z value shows up multiple times in different widths. This implies that the same fragment ion was detected across all 3 DIA cycles (width = -1,0, and 1), which is exactly the kind of contextual evidence we can gain from DIA that Cascadia is designed to exploit. It shows our model that this peak belongs to a real peptide, and isn't just noise.

That being said, let's continually keep in mind our motivation. A single MS2 spectrum in DIA is inherently ambiguous - it contains fragments from **all** precursors present in a given isolation window. By combining information across time (neighboring scans) and across MS levels (MS1 and MS2), the augmented spectrum lets us encode all of this context.

### 4.3 Building the Augmented Spectrum from Real DIA Data

Now that we've seen *why* the augmented spectrum exists, let's actually build one from the real DIA file we just loaded ([`DIA/E24484_Mag_1_4_50_3.mzML`](DIA/E24484_Mag_1_4_50_3.mzML)).

#### What the instrument is doing in this file

This mzML contains **767 scans**: 20 MS1 surveys and 747 MS2 fragmentation scans, grouped into cycles of 1 MS1 followed by 38 MS2 windows (`cycle_len = 39`). Each MS2 window is ~5 Da wide (±2.5 Da around a target m/z), with adjacent windows spaced ~4 Da apart so they overlap by ~1 Da.

Crucially, **38 windows is not enough to cover the full DIA m/z range in one cycle**. This instrument uses an *interleaved* acquisition scheme: it cycles through 4 different phase blocks (call them A / B / C / D), each of which covers a different ~150 Da swath of the m/z axis. Only every **4th** cycle fragments the same m/z region.

| Phase | m/z span | Role |
|-------|----------|------|
| A | 551.5 – 704.6 Da | low-mid precursors |
| B | 703.6 – 856.6 Da | **contains our target, 818.1 Da** |
| C | 855.6 – 1008.7 Da | high precursors |
| D | 399.4 – 552.5 Da | low precursors |

(These spans are taken directly from the file's `isolationWindow` metadata. See [`scripts/AugmentedManim.py`](scripts/AugmentedManim.py) for how they were measured.)

#### How that changes the width = ±1 rule

Because the instrument only revisits each phase every 4 cycles, a `width = ±1` neighbour is **not** the cycle immediately before or after, it's the next cycle *in which the same isolation window gets fragmented again*. For our 818.1 Da target (in Phase B), those cycles are **1, 5, 9, 13, 17, …** — separated by 4 cycles each, which is ~0.20 min (≈12 s) of retention time.

| Width | Cycle | Role |
|-------|-------|------|
| −1    | 1     | previous Phase B scan |
| 0     | 5     | central Phase B scan |
| +1    | 9     | next Phase B scan |

We pick the window centered at **m/z = 818.1** (covering 815.6 – 820.6 Da) because it has dense MS1 *and* MS2 peaks in all three cycles, which makes the augmented spectrum easy to inspect.

## Augmented Spectrum Construction Animation

Here is an animation of what we just talked through.

In [12]:
from IPython.display import Video
Video('Videos/AugmentedManim.mp4', embed=True, width=800)

The next cells just operationalize this: read the file, find those three MS2 scans plus their parent MS1 surveys, and stitch their peaks into a single augmented-spectrum DataFrame.

In [3]:
#@title Run this cell to read in the data

DIA_FILE = 'DIA/E24484_Mag_1_4_50_3.mzML'

with mzml.read(DIA_FILE) as f:
    scans = list(f)

ms1_indices = [i for i, s in enumerate(scans) if s['ms level'] == 1]
cycle_len = ms1_indices[1] - ms1_indices[0]

print(f'Total scans: {len(scans)}')
print(f'MS1 scans:   {len(ms1_indices)}')
print(f'Cycle length: {cycle_len} ({cycle_len - 1} MS2 + {1} MS1)')

Total scans: 767
MS1 scans:   20
Cycle length: 39 (38 MS2 + 1 MS1)


#### Finding the right MS2 scans

Isolating a single MS2 scan in DIA is a bit more involved than it sounds: we can't just index into `scans` by retention time, because every cycle has 38 MS2 scans and only *one* of them covers 818.1 Da. We also can't reuse a single fixed index across cycles — each cycle fragments phases in the same order, but absolute scan indices drift as the run progresses.

So we do it in two steps:

1. **Locate the cycle** — cycles are delimited by MS1 scans, so `ms1_indices[cycle_num]` and `ms1_indices[cycle_num + 1]` bound the MS2 scans belonging to that cycle.
2. **Scan the MS2s inside that cycle** for the one whose `isolation window target m/z` matches 818.1 (we allow a ±0.5 Da tolerance to be safe).

That's what `find_ms2_in_cycle` does below. `get_ms1_in_window` does the MS1 half — it takes the single MS1 survey that starts the cycle and slices out just the peaks whose m/z falls inside the 815.6–820.6 Da window.

In [4]:
# @title Run this cell to find MS2 scans matching the target isolation window and count MS1 peaks in the window

TARGET_IW = 818.1   # isolation window target m/z
IW_HALF   = 2.5     # window half-width

def find_ms2_in_cycle(cycle_num, target_isolation_window, scans, ms1_indices):
    """Return (index, scan) for the MS2 in cycle_num matching target_isolation_window."""
    ms1_idx      = ms1_indices[cycle_num] # index of the MS1 scan starting this cycle
    next_ms1_idx = ms1_indices[cycle_num + 1] # index of the next MS1 scan starting the next cycle
    for i in range(ms1_idx + 1, next_ms1_idx): # loop over MS2 scans in this cycle
        iw = scans[i]['precursorList']['precursor'][0]['isolationWindow'] # get isolation window info
        if abs(iw['isolation window target m/z'] - target_isolation_window) < 0.5:
            return i, scans[i]
    return None, None

def get_ms1_in_window(cycle_num, target_isolation_window, iw_half, scans, ms1_indices):
    """Return (mz, intensity) arrays for MS1 peaks inside the isolation window."""
    s   = scans[ms1_indices[cycle_num]]
    mz  = s.get('m/z array', np.array([]))
    int = s.get('intensity array', np.array([]))
    mask = (mz >= target_isolation_window - iw_half) & (mz <= target_isolation_window + iw_half)
    return mz[mask], int[mask]

# Cycles 1, 5, 9 all contain isolation window ~818.1
CYCLES = {-1: 1, 0: 5, 1: 9}   # map of {width : cycle_number}

for w, c in CYCLES.items():
    idx, scan = find_ms2_in_cycle(c, TARGET_IW, scans, ms1_indices)
    rt        = scan['scanList']['scan'][0]['scan start time']
    ms1_mz, _ = get_ms1_in_window(c, TARGET_IW, IW_HALF, scans, ms1_indices)
    print(f'width={w:+d}  cycle={c}  MS2_idx={idx}  rt={rt:.3f} min  '
          f'MS2_peaks={len(scan["m/z array"])}  MS1_in_window={len(ms1_mz)}')

width=-1  cycle=1  MS2_idx=69  rt=37.090 min  MS2_peaks=1931  MS1_in_window=49
width=+0  cycle=5  MS2_idx=225  rt=37.293 min  MS2_peaks=3402  MS1_in_window=64
width=+1  cycle=9  MS2_idx=381  rt=37.498 min  MS2_peaks=2475  MS1_in_window=92


In [5]:
def normalize(arr):
    mx = arr.max()
    return arr / mx if mx > 0 else arr

rows = []

for w, c in CYCLES.items():
    # take top 50 MS2 peaks by intensity
    _, ms2 = find_ms2_in_cycle(c, TARGET_IW, scans, ms1_indices)
    mz2  = ms2.get('m/z array', np.array([]))
    int2 = normalize(ms2.get('intensity array', np.array([])))
    top  = np.argsort(int2)[::-1][:50]
    for i in top:
        rows.append({'m/z': round(float(mz2[i]), 4),
                     'intensity': round(float(int2[i]), 4),
                     'width': w, 'level': 2})

    # MS1 peaks within isolation window
    ms1_mz, ms1_int = get_ms1_in_window(c, TARGET_IW, IW_HALF, scans, ms1_indices)
    if len(ms1_int) > 0:
        ms1_int_norm = normalize(ms1_int)
        for mz_val, int_val in zip(ms1_mz, ms1_int_norm):
            rows.append({'m/z': round(float(mz_val), 4),
                         'intensity': round(float(int_val), 4),
                         'width': w, 'level': 1})

aug_df = pd.DataFrame(rows)
print(f'Augmented spectrum: {len(aug_df)} total peaks')
print()
print(aug_df.groupby(['width', 'level']).size()
          .reset_index(name='peaks')
          .to_string(index=False))
display(aug_df)

Augmented spectrum: 355 total peaks

 width  level  peaks
    -1      1     49
    -1      2     50
     0      1     64
     0      2     50
     1      1     92
     1      2     50


,m/z,intensity,width,level
0,819.4313,1.0000,-1,2
1,820.4454,0.5515,-1,2
2,817.4212,0.3787,-1,2
3,818.4322,0.2317,-1,2
4,777.3968,0.0812,-1,2
...,...,...,...,...
350,820.5368,0.0026,1,1
351,820.5509,0.0013,1,1
352,820.5650,0.0000,1,1
353,820.5685,0.0000,1,1


### 4.4 Encoding Each 4-Tuple

We now have our augmented spectrum as a table of 4-tuples `(m/z, intensity, width, level)`. The next step is to convert each tuple into a fixed-size vector the transformer can operate on.

Recall that in Casanovo, each peak was described by just 2 numbers: m/z and intensity. Cascadia extends this to 4 numbers by adding **width** (which scan in the temporal window) and **level** (MS1 vs. MS2).

The encoding strategy for each component:

| Component | Value range | Encoding method |
|-----------|------------|------------------|
| m/z | continuous, ~100–2000 Da | Sinusoidal PE (same as Casanovo) |
| width | integer, {−1, 0, +1} | Sinusoidal PE (same formula) |
| intensity | continuous, [0, 1] | Learned linear layer |
| level | binary, {1, 2} | Learned linear layer |

The two sinusoidal encodings (m/z and width) are concatenated into $v_{\text{mz+width}}$. The two learned encodings (intensity and level) are trained alongside the rest of the model. The full per-peak vector is:

$$v_{\text{peak}} = [\,v_{\text{mz}} \;\|\; v_{\text{width}} \;\|\; v_{\text{int}} \;\|\; v_{\text{level}}\,]$$

*We can't simulate the learned encodings without training, but the sinusoidal ones are fully determined by the input, so below we can focus on visualizing the m/z and width encodings.*

In [6]:
# @title Run this cell to define the sinusoidal encoding functions
# this is the exact same formula from Casanovo.
def make_wavelength_list(dimensions, lambda_min=0.001, lambda_max=10000):
    """
    Generate wavelengths following the Casanovo formula:
    lambda_i = (lambda_min / 2pi) * (lambda_max / lambda_min)^(i / (d_sin - 1))
    """
    d_sin = dimensions // 2
    i = np.arange(d_sin)
    return (lambda_min / (2 * np.pi)) * (lambda_max / lambda_min) ** (i / (d_sin - 1))
# this is the exact same formula from Casanovo.
def sinusoidal_encoding(value, dimensions=512, lambda_min=0.001, lambda_max=10000):
    """
    Encode a scalar value into a d_model-dimensional sinusoidal vector.
    First half: sine encoding. Second half: cosine encoding.
    """
    wavelengths = make_wavelength_list(dimensions, lambda_min, lambda_max)
    d_sin = dimensions // 2
    enc = np.empty(dimensions)
    enc[:d_sin] = np.sin(value / wavelengths)
    enc[d_sin:] = np.cos(value / wavelengths)
    return enc

#### Encoding a single peak

Let's trace through the encoding for a real peak from our augmented spectrum.

We'll pick the **most intense fragment ion that falls inside the 815.6–820.6 Da isolation window** in the central MS2 scan (cycle 5, width = 0). For this file that peak is at **m/z ≈ 820.47**. 

True to the core ideas of DIA, this isn't a coincidence. The same ion shows up as the dominant peak in cycles 1, 5, and 9 (widths −1, 0, +1), which is exactly the recurring-across-widths signal the augmented spectrum is built to capture.

In [ ]:
# Pick the most intense MS2 fragment that falls inside the 815.6-820.6 Da
# isolation window of the central scan. This is the ion that recurs across
# widths -1/0/+1 in this mzML file.
_, central_scan = find_ms2_in_cycle(5, TARGET_IW, scans, ms1_indices)
mz2  = np.asarray(central_scan['m/z array'])
int2 = np.asarray(central_scan['intensity array'])

in_window  = (mz2 >= TARGET_IW - IW_HALF) & (mz2 <= TARGET_IW + IW_HALF)
masked_int = np.where(in_window, int2, -1.0)
top_i      = int(np.argmax(masked_int))

top_mz    = float(mz2[top_i])
top_int   = float(int2[top_i]) / float(int2.max())   # normalized to scan-wide max
top_width = 0   # central scan
top_level = 2   # MS2

print(f'Peak: m/z={top_mz:.4f}, normalized intensity={top_int:.4f}, width={top_width}, level={top_level}')
print()

enc_mz    = sinusoidal_encoding(top_mz)
enc_width = sinusoidal_encoding(top_width)

print(f'enc_mz    shape: {enc_mz.shape}')
print(f'enc_width shape: {enc_width.shape}')
print()
print(f'enc_mz    (dims 0-5):   {enc_mz[:6].round(4)}')
print(f'enc_width (dims 0-5):   {enc_width[:6].round(4)}')
print()
print('Note: width=0 encodes to sin(0/lambda)=0 and cos(0/lambda)=1 for every lambda.')
print('      That is the "origin" encoding; the central scan.')

#### Visualizing the Augmented Spectrum Encodings

The whole point of this notebook is that Cascadia's per-peak vector isn't $v_{\text{mz}}$ — it's $[\,v_{\text{mz}} \;\|\; v_{\text{width}}\,]$, stitched from *three* scans instead of one.

We'll show two plots that each expose one thing that makes the augmented spectrum different from a plain one:

1. **A single recurring fragment ion traced across all three widths.** We use the in-window peak at **m/z ≈ 820.47**, the same dominant ion that showed up in the central scan one section up, and which also appears as the strongest peak in the width = −1 and width = +1 scans. Its $v_{\text{mz}}$ column should be nearly identical in all three widths; its $v_{\text{width}}$ column should clearly differ. That's the contextual-evidence idea that Cascadia aims to exploit.

2. **MS1 vs. MS2 peaks partitioned by level.** A reminder that `level` is *learned*, not sinusoidally encoded, but the augmented spectrum still mixes both. MS1 peaks live strictly inside the 815.6–820.6 Da isolation window; MS2 fragments land anywhere below the precursor. Two independent views of the same chemical event.

#### (1) The 820.47 Da ion at all three widths

The m/z we just used in the single-peak example (≈ 820.47) is also the strongest in-window fragment in the width = −1 and width = +1 cycles. We locate it in each width group by picking the most-intense MS2 peak whose m/z rounds to 820.47 ± 0.05 Da, then plot the three $[v_{\text{mz}} \;\|\; v_{\text{width}}]$ vectors as adjacent columns.

In [ ]:
# Find the recurring ~820.47 Da MS2 peak at each width
TRACE_TARGET_MZ = 820.47   # the strongest in-window fragment in the central scan
TRACE_TOLERANCE = 0.05     # +/- Da window for matching the same ion across scans

ms2 = aug_df[aug_df['level'] == 2]
trace_rows = []
for w in (-1, 0, 1):
    sub  = ms2[ms2['width'] == w]
    near = sub[(sub['m/z'] - TRACE_TARGET_MZ).abs() <= TRACE_TOLERANCE]
    assert len(near) > 0, f'No MS2 peak near {TRACE_TARGET_MZ} Da at width {w}'
    trace_rows.append(near.sort_values('intensity', ascending=False).iloc[0])

trace = pd.DataFrame(trace_rows)
print(f'Tracking fragment m/z ~ {TRACE_TARGET_MZ} Da across widths:')
print(trace[['m/z', 'intensity', 'width', 'level']].to_string(index=False))

d_model = 512
cols, labels = [], []
for _, row in trace.iterrows():
    cols.append(np.concatenate([
        sinusoidal_encoding(float(row['m/z']),   dimensions=d_model),
        sinusoidal_encoding(float(row['width']), dimensions=d_model),
    ]))
    labels.append(f"width = {int(row['width']):+d}\nm/z = {float(row['m/z']):.4f}")

trace_mat = np.vstack(cols).T   # (1024, 3)

fig, ax = plt.subplots(figsize=(6, 7))
im = ax.imshow(trace_mat, aspect='auto', cmap='plasma', vmin=-1, vmax=1, interpolation='nearest')
plt.colorbar(im, ax=ax, label='Encoded value')
ax.axhline(d_model - 0.5, color='cyan', linewidth=2)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Encoding dimension (0-1023)')
ax.text(-0.8, d_model / 2,       'v_mz',    rotation=90, va='center', ha='right', color='white', fontsize=11)
ax.text(-0.8, d_model + d_model / 2, 'v_width', rotation=90, va='center', ha='right', color='cyan',  fontsize=11)
ax.set_title(f'(1) Same ion (m/z ~ {TRACE_TARGET_MZ}) at 3 widths\nTop half identical, bottom half differs')
plt.tight_layout()
plt.show()

#### (2) MS1 (level = 1) vs. MS2 (level = 2) peaks, same sinusoidal machinery

`level` is *learned*, not sinusoidally encoded, so we can't visualize its per-peak vector the same way. But we can still split the augmented spectrum by level and show that MS1 (precursor / isotope peaks inside the isolation window) and MS2 (fragment ions from anywhere in the scan) populate very different m/z regions, which is why Cascadia wants both: they're two independent views of the same chemical event.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

for ax, lvl, title in [(axes[0], 2, 'MS2 fragment peaks'),
                       (axes[1], 1, 'MS1 precursor/isotope peaks')]:
    sub = aug_df[aug_df['level'] == lvl]
    for w, color in zip([-1, 0, 1], ['#d62728', '#2ca02c', '#1f77b4']):
        s = sub[sub['width'] == w]
        ax.vlines(s['m/z'], 0, s['intensity'], color=color, alpha=0.7, lw=1.3, label=f'width = {w:+d}')
    ax.set_xlabel('m/z (Da)')
    ax.set_title(f'(2) {title}   n = {len(sub)}')
    ax.legend(fontsize=9)

axes[0].set_ylabel('Normalized intensity')
axes[1].axvspan(TARGET_IW - IW_HALF, TARGET_IW + IW_HALF, color='orange', alpha=0.15, label='isolation window')
axes[1].legend(fontsize=9)
plt.suptitle('MS1 peaks live inside the isolation window; MS2 fragments land anywhere below the precursor')
plt.tight_layout()
plt.show()